# Quantum-Simulated LLM — Google Colab (T4 GPU)
**World's first hybrid quantum-classical language model with agentic workflows**

Author: Saikrishna Garikipati

---
**Runtime → Change runtime type → T4 GPU** before running.

## 1. Setup

In [ ]:
# Clone repo and install
!git clone https://github.com/saikrishnajava/quantum-llm-agent.git
%cd quantum-llm-agent
!pip install -q pennylane pennylane-lightning numpy scipy pyyaml pytest
!pip install -q pennylane-lightning[gpu] custatevec-cu12 2>/dev/null || echo 'GPU packages failed, will use CPU fallback'
!pip install -q -e .

In [ ]:
# Verify GPU and backend
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader

import pennylane as qml
print(f'PennyLane: {qml.__version__}')

for dev_name in ['lightning.gpu', 'lightning.qubit', 'default.qubit']:
    try:
        dev = qml.device(dev_name, wires=4)
        @qml.qnode(dev)
        def test(): qml.Hadamard(0); return qml.expval(qml.PauliZ(0))
        test()
        print(f'{dev_name}: OK ✓')
    except Exception as e:
        print(f'{dev_name}: FAILED ({e})')

## 2. Run Tests

In [ ]:
!python -m pytest tests/ -v --tb=short 2>&1 | tail -20

## 3. Quick Benchmark — Compare Backends

In [ ]:
import sys, time, warnings
import numpy as np
sys.path.insert(0, '.')
warnings.filterwarnings('ignore')

from hybrid.interfaces.model import HybridQuantumLLM
from classical.optimizers.trainer import HybridQuantumTrainer
from classical.tokenizer import CharTokenizer
from classical.data import TextDataset, DataLoader

CORPUS = '''To be or not to be that is the question
Whether tis nobler in the mind to suffer
The slings and arrows of outrageous fortune
Or to take arms against a sea of troubles
And by opposing end them To die to sleep'''.strip()

tokenizer = CharTokenizer.from_text(CORPUS)
token_ids = tokenizer.encode(CORPUS)
dataset = TextDataset(token_ids, seq_len=8)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

model = HybridQuantumLLM(
    vocab_size=tokenizer.vocab_size, d_model=32, n_layers=1, n_heads=4,
    quantum_heads_per_layer=1, d_ff=64, max_seq_length=32, dropout=0.0,
    quantum_config={'use_quantum_embedding': False, 'embedding_qubits': 3,
                    'attention_qubits': 6, 'activation_qubits': 3,
                    'use_quantum_activation': False},
)
trainer = HybridQuantumTrainer(model, learning_rate=1e-3)

# Time 10 steps
times = []
for i, (x, y) in enumerate(loader):
    if i >= 10: break
    t0 = time.perf_counter()
    result = trainer.training_step(x, y)
    ms = (time.perf_counter() - t0) * 1000
    times.append(ms)
    print(f'Step {i+1}: loss={result["loss"]:.4f}, {ms:.0f}ms')

avg = np.mean(times[1:])  # skip warmup
print(f'\nColab T4 avg: {avg:.0f}ms/step')
print(f'Linux (lightning.qubit CPU): 2450ms/step')
print(f'MacBook (default.qubit): 3534ms/step')
print(f'Speedup vs Linux: {2450/avg:.1f}x')
print(f'Speedup vs Mac: {3534/avg:.1f}x')

## 4. Full Training — Shakespeare Corpus

In [ ]:
FULL_CORPUS = '''To be or not to be that is the question
Whether tis nobler in the mind to suffer
The slings and arrows of outrageous fortune
Or to take arms against a sea of troubles
And by opposing end them To die to sleep
No more and by a sleep to say we end
The heartache and the thousand natural shocks
That flesh is heir to Tis a consummation
Devoutly to be wished To die to sleep
To sleep perchance to dream ay there s the rub
For in that sleep of death what dreams may come
When we have shuffled off this mortal coil
Must give us pause There s the respect
That makes calamity of so long life
All that glitters is not gold
The fault dear Brutus is not in our stars
But in ourselves that we are underlings
Friends Romans countrymen lend me your ears
I come to bury Caesar not to praise him
The evil that men do lives after them
The good is oft interred with their bones
Now is the winter of our discontent
Made glorious summer by this sun of York
A horse a horse my kingdom for a horse
Out out brief candle life is but a walking shadow
A poor player that struts and frets his hour upon the stage
And then is heard no more it is a tale told by an idiot
Full of sound and fury signifying nothing
Double double toil and trouble fire burn and cauldron bubble
Something wicked this way comes
We are such stuff as dreams are made on
And our little life is rounded with a sleep'''.strip()

tokenizer = CharTokenizer.from_text(FULL_CORPUS)
token_ids = tokenizer.encode(FULL_CORPUS)
dataset = TextDataset(token_ids, seq_len=8)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

print(f'Corpus: {len(FULL_CORPUS)} chars, vocab: {tokenizer.vocab_size}, batches/epoch: {len(loader)}')

model = HybridQuantumLLM(
    vocab_size=tokenizer.vocab_size, d_model=32, n_layers=1, n_heads=4,
    quantum_heads_per_layer=1, d_ff=64, max_seq_length=32, dropout=0.0,
    quantum_config={'use_quantum_embedding': False, 'embedding_qubits': 3,
                    'attention_qubits': 6, 'activation_qubits': 3,
                    'use_quantum_activation': False},
)
params = model.count_parameters()
print(f'Model: {params["total"]:,} params ({params["quantum"]} quantum)')

trainer = HybridQuantumTrainer(model, learning_rate=1e-3)

N_EPOCHS = 10
print(f'\n--- Training ({N_EPOCHS} epochs) ---')
t_total = time.perf_counter()
epoch_losses = []

for epoch in range(N_EPOCHS):
    t0 = time.perf_counter()
    losses = []
    for x, y in loader:
        result = trainer.training_step(x, y)
        losses.append(result['loss'])
    elapsed = time.perf_counter() - t0
    avg_loss = np.mean(losses)
    epoch_losses.append(avg_loss)
    ms_step = elapsed / len(losses) * 1000
    print(f'  Epoch {epoch+1:2d}: loss={avg_loss:.4f} ({ms_step:.0f}ms/step, {elapsed:.1f}s)')

total_time = time.perf_counter() - t_total
print(f'\nTotal: {total_time:.1f}s ({total_time/60:.1f} min)')
print(f'Loss: {epoch_losses[0]:.4f} → {epoch_losses[-1]:.4f}')

## 5. Generate Text

In [ ]:
model.eval()
for prompt in ['To be', 'The fault', 'Now is', 'A horse']:
    ids = np.array([tokenizer.encode(prompt)])
    gen = model.generate(ids, max_new_tokens=60, temperature=0.7, do_sample=True)
    print(f'  "{prompt}" → "{tokenizer.decode(gen[0])}"')
    print()

## 6. Push T4 Limits — Bigger Model

In [ ]:
# Larger model: d_model=64, 2 quantum heads, quantum embedding
big_model = HybridQuantumLLM(
    vocab_size=tokenizer.vocab_size, d_model=64, n_layers=2, n_heads=8,
    quantum_heads_per_layer=2, d_ff=256, max_seq_length=64, dropout=0.0,
    quantum_config={'use_quantum_embedding': True, 'embedding_qubits': 4,
                    'attention_qubits': 6, 'activation_qubits': 3,
                    'use_quantum_activation': False},
)
big_params = big_model.count_parameters()
print(f'Big model: {big_params["total"]:,} params ({big_params["quantum"]} quantum)')
print(f'Quantum components: embedding + 2 attention heads per layer × 2 layers')

dataset_big = TextDataset(token_ids, seq_len=16)
loader_big = DataLoader(dataset_big, batch_size=4, shuffle=True)

big_trainer = HybridQuantumTrainer(big_model, learning_rate=5e-4)

N_EPOCHS = 10
print(f'\n--- Big Model Training ({N_EPOCHS} epochs, {len(loader_big)} batches/epoch) ---')
t_total = time.perf_counter()

for epoch in range(N_EPOCHS):
    t0 = time.perf_counter()
    losses = []
    for x, y in loader_big:
        result = big_trainer.training_step(x, y)
        losses.append(result['loss'])
    elapsed = time.perf_counter() - t0
    ms_step = elapsed / len(losses) * 1000
    print(f'  Epoch {epoch+1:2d}: loss={np.mean(losses):.4f} ({ms_step:.0f}ms/step, {elapsed:.1f}s)')

total_time = time.perf_counter() - t_total
print(f'\nTotal: {total_time:.1f}s ({total_time/60:.1f} min)')

In [ ]:
# Generate with the big model
big_model.eval()
for prompt in ['To be or not', 'The fault dear', 'Something wicked']:
    ids = np.array([tokenizer.encode(prompt)])
    gen = big_model.generate(ids, max_new_tokens=80, temperature=0.7, do_sample=True)
    print(f'  "{prompt}" → "{tokenizer.decode(gen[0])}"')
    print()

## 7. Max Scale — T4 Stress Test

In [ ]:
# Maximum model the T4 can handle
# d_model=128, 4 quantum heads, quantum embedding, 4 layers
max_model = HybridQuantumLLM(
    vocab_size=tokenizer.vocab_size, d_model=128, n_layers=4, n_heads=8,
    quantum_heads_per_layer=2, d_ff=512, max_seq_length=64, dropout=0.0,
    quantum_config={'use_quantum_embedding': True, 'embedding_qubits': 6,
                    'attention_qubits': 6, 'activation_qubits': 4,
                    'use_quantum_activation': False},
)
max_params = max_model.count_parameters()
print(f'Max model: {max_params["total"]:,} params ({max_params["quantum"]} quantum)')

# Forward pass timing
test_ids = np.random.randint(0, tokenizer.vocab_size, (1, 16))
t0 = time.perf_counter()
logits = max_model(test_ids)
fwd_ms = (time.perf_counter() - t0) * 1000
print(f'Forward pass: {fwd_ms:.0f}ms, output shape: {logits.shape}')

# Single training step
max_trainer = HybridQuantumTrainer(max_model, learning_rate=1e-4)
x = np.random.randint(0, tokenizer.vocab_size, (2, 16))
y = np.random.randint(0, tokenizer.vocab_size, (2, 16))
t0 = time.perf_counter()
result = max_trainer.training_step(x, y)
train_ms = (time.perf_counter() - t0) * 1000
print(f'Training step: {train_ms:.0f}ms, loss={result["loss"]:.4f}, grad_norm={result["grad_norm"]:.4f}')
print(f'\nThis is the largest model we can train with quantum circuits on T4.')

---
**Copyright (c) 2026 Saikrishna Garikipati. All Rights Reserved.**